# MegaLoc + OPR_GraphEnhancedMegaloc64 rerank

Запуск `gsloc.inference.test.Test`:
- **model** (первичный поиск): `MegaLoc`
- **rerank_model**: `OPR_GraphEnhancedMegaloc64` (`graph_model64` из `09_GTgraph_all.ipynb`)

In [ ]:
from gsloc.inference.test import TestConfig, Test
from pathlib import Path
from gsloc.models import opr_graph_extention as network
import torch
from torchvision.transforms import functional as F
from mmpr.models import MegaLoc
from gsloc.datasets import ThreeRScan
from torchvision import transforms as T

import random
import numpy as np

In [ ]:
def make_deterministic(seed=0):
    if seed == -1:
        return
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

make_deterministic(0)

In [ ]:
similarity_kwargs_list = [
    {"mode": "room", "trans_tol_m": 3, "rot_tol_deg": 180},
    {"mode": "pose", "trans_tol_m": 3, "rot_tol_deg": 180},
    {"mode": "pose", "trans_tol_m": 2, "rot_tol_deg": 90},
]

seq_filter_kwargs_list = [
    {
        "seq_similarity_filter_mode": "none",
        "seq_similarity_trans_tol_m": 0.5,
        "seq_similarity_rot_tol_deg": 15,
    },
    {
        "seq_similarity_filter_mode": "pose",
        "seq_similarity_trans_tol_m": 0.5,
        "seq_similarity_rot_tol_deg": 15,
    },
    {
        "seq_similarity_filter_mode": "pose",
        "seq_similarity_trans_tol_m": 1,
        "seq_similarity_rot_tol_deg": 30,
    },
]

image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Resize([322, 322], antialias=True),
    T.Lambda(lambda x: F.rotate(x, angle=-90)),
    T.Normalize(
        mean=[0.44420420130352495, 0.41322746532289134, 0.3678658064565412],
        std=[0.24352604373543688, 0.24045797651069503, 0.24250136992133814],
    ),
])

In [ ]:
weights_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatV3/best_model.pth")
ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GAT_graph_encoder = network.OPR_GATGraphEncoder64(
    in_dim=4,
    hidden_dim=512,
    n_layers=1,
    num_node_classes=529,
    node_emb_dim=64,
    num_edge_classes=41,
    edge_emb_dim=128,
    proj_dim=64,
    edge_cont_dim=10,
    dropout=0.1,
    heads=4,
).to(device)

graph_model64 = network.OPR_GraphEnhancedMegaloc64(
    graph_encoder=GAT_graph_encoder,
    image_encoder=None,
)

missing, unexpected = graph_model64.load_state_dict(ckpt["multimodal_state_dict"], strict=False)
ignored_unexpected_prefixes = ("image_encoder.", "graph_encoder.convs.")
unexpected_other = [k for k in unexpected if not k.startswith(ignored_unexpected_prefixes)]
if unexpected_other:
    raise RuntimeError(f"Unexpected checkpoint keys: {unexpected_other}")

graph_model64.to(device)
graph_model64.eval()

In [ ]:
megaLoc = MegaLoc()
megaLoc.to(device)
megaLoc.eval()

In [ ]:
def run_megaloc_graph64_test(
    tests_path,
    dataset_path,
    dataset_name,
    date,
    graph_type,
    graphmodel_type,
    image_model_type,
    rerank_k,
    per_frame_k,
    filter_type,
    similarity_type,
    graph_dir,
    edge_normalizer_path,
    scene_list_path,
    query_list_path,
    room_json_path,
    seq_filter_kwargs,
    similarity_kwargs,
    graph_model,
    image_model,
):
    """MegaLoc — первичный индекс; graph_model — реранжирование (обратно run_test в 09_GTgraph_all)."""
    model_name = f"{image_model_type}x{graphmodel_type}"
    today_dataset_test_path = tests_path / date / dataset_name
    test_path = today_dataset_test_path / graph_type / model_name

    index_path = today_dataset_test_path / "cache" / "indexes" / image_model_type
    query_cache_path = today_dataset_test_path / "cache" / "query_cache" / image_model_type
    rerank_index_path = (
        today_dataset_test_path / "cache" / "indexes" / graph_type / (graphmodel_type + "graph")
    )
    rerank_query_cache_path = (
        today_dataset_test_path / "cache" / "query_cache" / graph_type / (graphmodel_type + "graph")
    )
    frames_path = test_path / (f"rerank_k_{rerank_k}_per_frame_k_{per_frame_k}") / "frames.npz"
    bench_report_path = (
        test_path / (f"rerank_k_{rerank_k}_per_frame_k_{per_frame_k}") / filter_type / similarity_type
    )

    cfg = TestConfig(
        dataset_path=dataset_path,
        test_path=test_path,
        index_path=index_path,
        rerank_index_path=rerank_index_path,
        query_cache_path=query_cache_path,
        rerank_query_cache_path=rerank_query_cache_path,
        bench_report_path=bench_report_path,
        graph_path=graph_dir,
        dataset_class=ThreeRScan,
        filter_kwargs={"similarity_filter_mode": "none"},
        seq_filter_kwargs=seq_filter_kwargs,
        scene_list_path=scene_list_path,
        query_list_path=query_list_path,
        room_json_path=room_json_path,
        edge_normalizer_path=edge_normalizer_path,
        image_transform_fn=image_transform_fn,
        graph_feat_dim=4,
        graph_edge_attr_dim=10,
        graph_rotate=True,
        device=device,
        batch_size=16,
        num_workers=4,
        model=image_model,
        rerank_model=graph_model,
        rerank_k=rerank_k,
        per_frame_k_used=per_frame_k,
        final_k=25,
        seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
        recall_at_k=[1, 5, 10, 25],
        similarity_kwargs=similarity_kwargs,
        std_mode="global",
        scene_df_field="scene",
        pose_df_field="pose",
        frames_path=frames_path,
    )

    test = Test(cfg)
    test.run()

In [ ]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-05-19"
dataset_name = "3RScan"
graph_type = "GT"
graphmodel_type = "64"
image_model_type = "Megaloc"

rerank_k = 1000
per_frame_k = 25

filter_type = "base_seq_report"
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[0]
similarity_kwargs = similarity_kwargs_list[0]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"
query_list_path = None
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

In [ ]:
run_megaloc_graph64_test(
    tests_path=tests_path,
    dataset_path=dataset_path,
    dataset_name=dataset_name,
    date=date,
    graph_type=graph_type,
    graphmodel_type=graphmodel_type,
    image_model_type=image_model_type,
    rerank_k=rerank_k,
    per_frame_k=per_frame_k,
    filter_type=filter_type,
    similarity_type=similarity_type,
    graph_dir=graph_dir,
    edge_normalizer_path=edge_normalizer_path,
    scene_list_path=scene_list_path,
    query_list_path=query_list_path,
    room_json_path=room_json_path,
    seq_filter_kwargs=seq_filter_kwargs,
    similarity_kwargs=similarity_kwargs,
    graph_model=graph_model64,
    image_model=megaLoc,
)